# 3.5. Concise Implementation of Linear Regression
D2L의 Concise Implementation of Linear Regression장을 PyTorch 기준으로 정리함.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [ ]:
%matplotlib inline
import random
import torch
import matplotlib.pyplot as plt
from torch.distributions.multinomial import Multinomial
import os
from torch import nn

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

torch.manual_seed(0)
random.seed(0)

print("PyTorch version:", torch.__version__)

true_w = torch.tensor([[2.0], [-3.4]])  # [2, 1]
true_b = 4.2

num_examples = 1000

X = torch.randn(num_examples, 2)        # [1000, 2]
noise = torch.randn(num_examples, 1) * 0.01
y = X @ true_w + true_b + noise         # [1000, 1]

## 1. 이 장의 의미

Concise Implementation of Linear Regression
선형회구의 간결한 구현

PyTorch 내장 기능으로 바꾸는 것을 배운다.

아래는 예시이다.

In [ ]:
w = torch.normal(0, 0.01, size=(2, 1), requires_grad=True)
b = torch.zeros(1, requires_grad=True)

def linreg(X, w, b):
    return X @ w + b

def squared_loss(y_hat, y):
    return (y_hat - y.reshape(y_hat.shape)) ** 2 / 2

def sgd(params, lr, batch_size):
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad / batch_size
            param.grad.zero_()

In [ ]:
model = nn.Linear(2, 1) # 내부에서 하는 건 거의 같다
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.03)

## 2. 모델 정의 : `nn.Linear`

D2L 원문에서는 PyTorch의 fully connected layer가 Linear 또는 LazyLinear로 정의된다고 설명했다. LazyLinear는 출력 차원만 지정하고 입력차원은 첫 forward 때 자동 추론하는 방식이다. D2L 예시는 nn.LazyLinear(1)로 출력 하나짜리 선형계층을 만든다.

In [ ]:
model = nn.Linear(2, 1) # 이건 내부적으로 w: [1,2], b: [1]을 만든다.

# y_hat = X @ w + b

print("model.weight.shape :", model.weight.shape) # 왜 [2, 1]이 아니고 [1, 2]인가
print("model.bias.shape :", model.bias.shape)

# PyTorch 내부 계산은 이렇게 동작한다고한다.
# y_hat = X @ model.weight.T + model.bias

## 3. forward 입력을 모델에 넣으면 예측값이 나온다.

D2L에서는 forward(self, X) 안에서 그냥 self.net(X)를 호출한다. 직접 X @ w + b를 쓰지 않고 layer에 계산을 맡긴다.

In [ ]:
X = torch.randn(5, 2) # 입력데이터 5개 feature는 2개씩

y_hat = model(X)

print(y_hat.shape) # [5,1] 각 데이터마다 예측값 1개씩

## 4. 손실 함수: nn.MSELoss

D2L에서는 MSELoss가 mean squared error를 계산한다고 설명했다. 원래 D2L 앞장 squared loss에는 1/2가 들어갔는데 PyTorch MSELoss는 기본적으로 1/2 없이 계산한다. 기본값은 example 전체에 대한 평균 loss이다.

In [ ]:
y_hat = model(X)
loss_fn = nn.MSELoss()
loss = loss_fn(y_hat, y)

print("y_hat.shape :", y_hat.shape) # [~, 1]로 맞춰야한다.
print("y.shape", y.shape)

## 5. Optimizer: torch.optim.SGD

D2L에서는 PyTorch의 optim모듈이 minibatch SGD와 여러 변형 optimizer를 제공한다고 설명한다. torch.optim.SGD(self.parameters(), self.lr)처럼 모델 파라미터와 learning rate를 넘겨서 optimizer를 만든다.

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.03)

optimizer.zero_grad() # gradient 제거
loss.backward() # loss기준으로 w, b의 gradient 계산
optimizer.step() # w, b 업데이트
# SGD는 계산된 gradient보고 parameter 업데이트하는 알고리즘임.

## 6. 전체 코드

In [ ]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

# 정답 파라미터
true_w = torch.tensor([[2.0], [-3.4]])  # [2, 1]
true_b = 4.2

# 데이터 개수
num_examples = 1000

# 입력 X: [1000, 2]
X = torch.randn(num_examples, 2)

# noise: 현실 데이터처럼 오차 추가
noise = torch.randn(num_examples, 1) * 0.01

# 정답 y: [1000, 1]
y = X @ true_w + true_b + noise

print(X.shape)
print(y.shape)

# Dataset과 DataLoader
dataset = TensorDataset(X, y)

batch_size = 10
data_iter = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# 모델: feature 2개 => 예측값 1개
model = nn.Linear(2, 1)

# 파라미터 초기화
model.weight.data.normal_(0, 0.01)
model.bias.data.fill_(0)

print(model.weight)
print(model.bias)

# 손실 함수
loss_fn = nn.MSELoss()

# optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=0.03)

num_epochs = 3

for epoch in range(num_epochs):
    for X_batch, y_batch in data_iter:
        # 1. 예측
        y_hat = model(X_batch)

        # 2. loss 계산
        loss = loss_fn(y_hat, y_batch)

        # 3. 이전 gradient 제거
        optimizer.zero_grad()

        # 4. gradient 계산
        loss.backward()

        # 5. parameter 업데이트
        optimizer.step()

    with torch.no_grad():
        train_loss = loss_fn(model(X), y)
        print(f"epoch {epoch + 1}, loss {train_loss.item():.6f}")

print("true_w:", true_w.reshape(-1))
print("learned_w:", model.weight.data.reshape(-1))

print("true_b:", true_b)
print("learned_b:", model.bias.data)

## 7. 중요한거

nn.Linear       # w, b 생성 + y_hat 계산   
nn.MSELoss      # loss 계산  
torch.optim.SGD # w, b 업데이트  
DataLoader      # minibatch 생성  

### 흐름
입력 X  
-> model(X)  
-> y_hat  
-> loss_fn(y_hat, y)  
-> loss  
-> loss.backward()  
-> optimizer.step()  
-> w, b 갱신  

## 8. 오늘의 정리

- 이번 장은 선형회귀를 직접 구현하지 않고 PyTorch의 고수준 API로 간결하게 구현하는 내용이다.
- 달라진 점은 `w`, `b`, loss, SGD를 직접 만들지 않고 PyTorch가 제공하는 도구를 쓴다는 것이다.

- `nn.Linear(in_features, out_features)`는 입력 feature 개수와 출력 개수를 지정하는 선형 계층이다.
- `nn.Linear(2, 1)`은 feature 2개짜리 입력을 받아 예측값 1개를 만든다는 뜻이다.
- 이 경우 결과적으로 weight는 2개, bias는 1개가 생긴다.
- `nn.Linear(5, 1)`은 feature 5개짜리 입력을 받아 예측값 1개를 만든다는 뜻이다.
- 이 경우 결과적으로 weight는 5개, bias는 1개가 생긴다.

- 일반 규칙은 다음과 같다.
  - `weight 개수 = in_features × out_features`
  - `bias 개수 = out_features`

- PyTorch의 `nn.Linear(2, 1)`에서 `model.weight.shape`은 `[1, 2]`이다.
- 이유는 PyTorch가 weight를 `[out_features, in_features]` 형태로 저장하기 때문이다.
- 하지만 실제 계산은 `X @ model.weight.T + model.bias`처럼 이해하면 된다.

- 입력 데이터 `X`의 shape이 `[batch_size, 2]`라면 각 데이터는 feature 2개를 가진다.
- `batch_size`는 한 번에 처리하는 데이터 개수이고, 모델 구조를 결정하지 않는다.
- 모델 구조를 결정하는 것은 feature 개수와 출력 개수다.

- `model(X)`는 모델에 입력 `X`를 넣어 예측값 `y_hat`을 만드는 과정이다.
- `y_hat`은 모델의 예측값이고, `y`는 정답값이다.
- loss를 계산하려면 `y_hat`과 `y`가 둘 다 있어야 한다.

- `nn.MSELoss()`는 예측값과 정답값의 평균 제곱 오차를 계산하는 손실 함수다.
- 선형회귀에서는 보통 `MSELoss`를 사용한다.
- `loss = loss_fn(y_hat, y)`는 모델 예측값과 정답값이 얼마나 다른지 계산하는 코드다.

- `loss.backward()`는 gradient를 계산한다.
- `optimizer.step()`은 계산된 gradient를 사용해서 weight와 bias를 업데이트한다.
- `optimizer.zero_grad()`는 이전 batch에서 쌓인 gradient를 초기화한다.
- gradient는 자동으로 덮어씌워지는 것이 아니라 누적되므로 매 반복마다 `zero_grad()`가 필요하다.

- 학습 루프의 기본 순서는 다음과 같다.
  1. `y_hat = model(X_batch)`로 예측한다.
  2. `loss = loss_fn(y_hat, y_batch)`로 손실을 계산한다.
  3. `optimizer.zero_grad()`로 이전 gradient를 지운다.
  4. `loss.backward()`로 gradient를 계산한다.
  5. `optimizer.step()`으로 parameter를 업데이트한다.

- `NameError: name 'y' is not defined`는 `y`라는 정답 변수를 만들기 전에 사용했다는 뜻이다.
- loss를 계산하려면 먼저 `X`, `y`, `model`, `y_hat`이 정의되어 있어야 한다.
- 셀을 중간부터 실행하면 변수가 정의되지 않아 이런 에러가 자주 난다.

- 이번 장에서 중요한 것은 코드를 짧게 쓰는 것이 아니라, 짧아진 코드 안에서 무슨 일이 일어나는지 아는 것이다.
- `nn.Linear`는 `w`, `b`를 만들고 선형 계산을 수행한다.
- `nn.MSELoss`는 예측값과 정답값의 차이를 loss로 계산한다.
- `torch.optim.SGD`는 gradient를 사용해서 `w`, `b`를 업데이트한다.

- 직접 구현과 간결한 구현은 겉모습만 다르다.
- 내부 흐름은 여전히 `입력 -> 예측 -> loss 계산 -> gradient 계산 -> parameter 업데이트`이다.